# Install and import packages
--------

In [1]:
# install/import quantum gym environments
#!pip install git+https://github.com/qdevpsi3/quantum-arch-search.git

# install/import stable baselines 3
#!pip install stable_baselines3

In [1]:
#import gym
import gymnasium as gym
import torch.optim as optim
from stable_baselines3 import A2C, PPO,DQN
from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import SAC
from stable_baselines3 import HerReplayBuffer
import qas_gym
from qas_gym import envs
import sys
import os

# Basic Environment
------
Create your gym environment :

In [2]:
# Parameters 
env_name = 'BasicThreeQubit-v0'#'NoisyThreeQubit-v0'#'BasicThreeQubit-v0'
#gym.wrappers.EnvCompatibility(env_name)
fidelity_threshold = 0.95
reward_penalty = 0.01
max_timesteps = 20#20
target = (1/(2*np.sqrt(2.))) * np.asarray([1, 1, 1, -1, 1, 1, -1, 1])
#target = np.zeros(2**3, dtype=complex)
#target[0] = 1. / np.sqrt(2) + 0.j
#target[-1] = 1. / np.sqrt(2) + 0.j

#seed
seed=2

# Environment
env = gym.make(env_name,
               target=target,
               fidelity_threshold=fidelity_threshold,
               reward_penalty=reward_penalty,
               max_timesteps=max_timesteps,#error_single=0.04, error_multi=0.05,error_rate = 0
              )
#env.observation_space.shape
#gym.wrappers.EnvCompatibility(env_name)
observation, info = env.reset(seed=seed)
print(env.action_space) 
print(env.observation_space)


Discrete(33)
Dict('entanglement': Box(0.0, 1.0, (1,), float32), 'expectations': Box(-1.0, 1.0, (9,), float32), 'gate_count': Discrete(21))


Diplay the action gates : 

In [3]:
for idx, gate in enumerate(env.unwrapped.action_gates):
    print('Action({:02d}) --> {}'.format(idx, gate))

Action(00) --> Rz(0.25π)(q(0))
Action(01) --> X(q(0))
Action(02) --> Y(q(0))
Action(03) --> Z(q(0))
Action(04) --> H(q(0))
Action(05) --> Ry(0.5π)(q(0))
Action(06) --> S(q(0))
Action(07) --> Rz(0.25π)(q(1))
Action(08) --> X(q(1))
Action(09) --> Y(q(1))
Action(10) --> Z(q(1))
Action(11) --> H(q(1))
Action(12) --> Ry(0.5π)(q(1))
Action(13) --> S(q(1))
Action(14) --> Rz(0.25π)(q(2))
Action(15) --> X(q(2))
Action(16) --> Y(q(2))
Action(17) --> Z(q(2))
Action(18) --> H(q(2))
Action(19) --> Ry(0.5π)(q(2))
Action(20) --> S(q(2))
Action(21) --> CNOT(q(0), q(1))
Action(22) --> CZ(q(0), q(1))
Action(23) --> CNOT(q(0), q(2))
Action(24) --> CZ(q(0), q(2))
Action(25) --> CNOT(q(1), q(0))
Action(26) --> CZ(q(1), q(0))
Action(27) --> CNOT(q(1), q(2))
Action(28) --> CZ(q(1), q(2))
Action(29) --> CNOT(q(2), q(0))
Action(30) --> CZ(q(2), q(0))
Action(31) --> CNOT(q(2), q(1))
Action(32) --> CZ(q(2), q(1))


Diplay the state observables : 

In [4]:
for idx, observable in enumerate(env.unwrapped.state_observables):
    print('State({:02d}) --> {}'.format(idx, observable))

State(00) --> X(q(0))
State(01) --> Y(q(0))
State(02) --> Z(q(0))
State(03) --> X(q(1))
State(04) --> Y(q(1))
State(05) --> Z(q(1))
State(06) --> X(q(2))
State(07) --> Y(q(2))
State(08) --> Z(q(2))


# PPO Model
------

In [ ]:
# Parameters
gamma = 0.97#0.97
n_epochs = 4
clip_range = 0.2
learning_rate = 0.0004#0.0004
policy_kwargs = dict(optimizer_class=optim.Adam)


# Agent
ppo_model = PPO("MultiInputPolicy",
                env,
                verbose=1,
                #batch_size=128,
                seed=seed,
                gamma=gamma,
                n_epochs=n_epochs,
                clip_range=clip_range,
                learning_rate=learning_rate,
                policy_kwargs=policy_kwargs,
                tensorboard_log='logs/')

In [ ]:
ppo_model.learn(total_timesteps=180000, progress_bar=True)#180000

In [ ]:
import time
from IPython.display import clear_output
import gymnasium as gym

# 
# create env first
#env = gym.make("...")
#from stable_baselines3 import PPO
#ppo_model = PPO.load("path/")

state, info = env.reset()
print("Initial state:",state) # Debug: Print initial state

done = False
truncated = False
entanglement_list_pre = []
fidelity_list_pre = []

while not done and not truncated:
    action, _states = ppo_model.predict(state, deterministic=False) 
    next_state, reward, done, truncated, info = env.step(action)

    entanglement_list_pre.append(env.unwrapped.entangle_list)
    fidelity_list_pre.append(env.unwrapped.fidelity_list)
    
    clear_output(wait=True)
    env.render()
    time.sleep(1)

    state = next_state   # Correctly update state
    print("Current state:", state) # Debug: Print current state

print(info)

In [7]:
ppo_model.save("ppo_model")

In [ ]:
model = PPO.load("ppo_model", env=env)

# 继续训练
#model.learn(total_timesteps=10000)

In [ ]:
ppo_model.learn(total_timesteps=20000, reset_num_timesteps=False)

# Results tensorboard
------

In [6]:
%load_ext tensorboard
%tensorboard --logdir=logs/

TypeError: QASPPO.__init__() missing 4 required positional arguments: 'M', 'S', 'U', and 'n_steps'